In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df=sns.load_dataset('penguins')

In [8]:
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


In [9]:
df.isnull().sum()

,0
species,0
island,0
bill_length_mm,2
bill_depth_mm,2
flipper_length_mm,2
body_mass_g,2
sex,11


In [10]:
df=df.dropna()

In [11]:
df.duplicated().sum()

np.int64(0)

In [12]:
df['species'].value_counts()

,count
species,
Adelie,146
Gentoo,119
Chinstrap,68


In [13]:
encode= pd.get_dummies(df,columns=['island','sex'],drop_first=True)

In [14]:
encode

,species,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,island_Dream,island_Torgersen,sex_Male
0,Adelie,39.1,18.7,181.0,3750.0,False,True,True
1,Adelie,39.5,17.4,186.0,3800.0,False,True,False
2,Adelie,40.3,18.0,195.0,3250.0,False,True,False
4,Adelie,36.7,19.3,193.0,3450.0,False,True,False
5,Adelie,39.3,20.6,190.0,3650.0,False,True,True
...,...,...,...,...,...,...,...,...
338,Gentoo,47.2,13.7,214.0,4925.0,False,False,False
340,Gentoo,46.8,14.3,215.0,4850.0,False,False,False
341,Gentoo,50.4,15.7,222.0,5750.0,False,False,True
342,Gentoo,45.2,14.8,212.0,5200.0,False,False,False


In [15]:
encode.isnull().sum()

,0
species,0
bill_length_mm,0
bill_depth_mm,0
flipper_length_mm,0
body_mass_g,0
island_Dream,0
island_Torgersen,0
sex_Male,0


In [16]:
bool_col = encode.select_dtypes('bool').columns
encode[bool_col] = encode[bool_col].astype(int)

In [17]:
encode['species']=encode['species'].map({'Adelie':1,'Gentoo':2,'Chinstrap':3})

In [19]:
x= encode.drop(columns=['species'])
y= encode['species']

In [32]:
x

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,island_Dream,island_Torgersen,sex_Male
0,39.1,18.7,181.0,3750.0,0,1,1
1,39.5,17.4,186.0,3800.0,0,1,0
2,40.3,18.0,195.0,3250.0,0,1,0
4,36.7,19.3,193.0,3450.0,0,1,0
5,39.3,20.6,190.0,3650.0,0,1,1
...,...,...,...,...,...,...,...
338,47.2,13.7,214.0,4925.0,0,0,0
340,46.8,14.3,215.0,4850.0,0,0,0
341,50.4,15.7,222.0,5750.0,0,0,1
342,45.2,14.8,212.0,5200.0,0,0,0


In [31]:
y

,species
0,1
1,1
2,1
4,1
5,1
...,...
338,2
340,2
341,2
342,2


In [21]:
encode

,species,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,island_Dream,island_Torgersen,sex_Male
0,1,39.1,18.7,181.0,3750.0,0,1,1
1,1,39.5,17.4,186.0,3800.0,0,1,0
2,1,40.3,18.0,195.0,3250.0,0,1,0
4,1,36.7,19.3,193.0,3450.0,0,1,0
5,1,39.3,20.6,190.0,3650.0,0,1,1
...,...,...,...,...,...,...,...,...
338,2,47.2,13.7,214.0,4925.0,0,0,0
340,2,46.8,14.3,215.0,4850.0,0,0,0
341,2,50.4,15.7,222.0,5750.0,0,0,1
342,2,45.2,14.8,212.0,5200.0,0,0,0


In [22]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score , classification_report

In [23]:
x_train, x_test , y_train, y_test= train_test_split(x,y,test_size=0.20,random_state=42)

In [24]:
from sklearn.preprocessing import StandardScaler
scaler= StandardScaler()
x_train_scale=scaler.fit_transform(x_train)
x_test_scale=scaler.transform(x_test)

In [25]:
base_leaners=[('dt', DecisionTreeClassifier(random_state=42)),('svc',SVC(probability=True,kernel='rbf',random_state=42)),('lr', LogisticRegression(max_iter=1000))]

In [26]:
meta_learner = LogisticRegression(max_iter=1000)

In [27]:
stacking_clf = StackingClassifier(estimators=base_leaners, final_estimator=meta_learner,cv=5)

In [28]:
stacking_clf.fit(x_train_scale,y_train)

StackingClassifier(cv=5,
                   estimators=[('dt', DecisionTreeClassifier(random_state=42)),
                               ('svc', SVC(probability=True, random_state=42)),
                               ('lr', LogisticRegression(max_iter=1000))],
                   final_estimator=LogisticRegression(max_iter=1000))

In [29]:
y_pred = stacking_clf.predict(x_test_scale)

In [30]:
accuracy_score(y_test,y_pred)

1.0